# QuEra Analog Hamiltonian Simulation

Place neutral atoms in a geometry, drive them with time-dependent fields, and let the Rydberg Hamiltonian solve Maximum Independent Set -- no gate circuit in sight.

**Objectives:**
- Contrast *analog* quantum computation (QuEra Aquila) with the gate model of IonQ and IQM.
- Build a real `AnalogHamiltonianSimulation` program with `braket.ahs`: an `AtomArrangement` plus a time-dependent `DrivingField` (Rabi frequency, detuning, phase).
- Understand the *Rydberg blockade* and how atom geometry encodes a graph whose Maximum Independent Set the adiabatic sweep prepares.
- Inspect the program's register coordinates and field schedule locally, and cross-check the answer with a classical (numpy) MIS solve.
- Keep the actual Aquila submission behind `RUN_ON_AWS = False` with a cost note, per the project's local-first discipline.

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

## 0. Analog vs. gate model

IonQ and IQM are *gate-model* machines: you compose discrete gates into a circuit. QuEra **Aquila** is different. It is a 256-atom array of rubidium atoms held in optical tweezers, and it does **not** run gate circuits at all. Instead you:

1. **Place** the atoms in a 2D geometry (the *register*).
2. **Drive** them with global time-dependent fields -- a Rabi frequency $\Omega(t)$ that couples each atom's ground state $\ket{g}$ to a highly excited *Rydberg* state $\ket{r}$, and a detuning $\delta(t)$ that tilts the energy landscape.
3. **Let the system evolve** under the Rydberg Hamiltonian and read out which atoms ended up in $\ket{r}$.

The physics that makes this useful is the **Rydberg blockade**: two atoms closer than a *blockade radius* $R_b$ cannot both be excited to $\ket{r}$ at once -- exciting one shifts the other's energy out of resonance. So if you draw an edge between every pair of atoms within $R_b$, the set of simultaneously-excited atoms is always an **independent set** of that graph. Drive the system adiabatically and it relaxes toward the *largest* such set: the **Maximum Independent Set (MIS)**. The geometry *is* the program.

This notebook builds that program object for real with `braket.ahs`, inspects its structure, and checks the MIS classically. It runs locally in a venv with the real Amazon Braket SDK but **no AWS credentials** -- so we only ever *construct* and *inspect*; the single line that would submit to Aquila is guarded behind `RUN_ON_AWS = False`.

In [ ]:
# Setup: run this cell first.
# Requires the real amazon-braket-sdk (pip install -e '.[dev]' from the project root; see `make setup`).
# Importing braket.ahs and braket.aws is free and offline -- it does NOT contact AWS.
# We do NOT instantiate AwsDevice or call get_devices() at top level; any real call is guarded later.
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations, product

from braket.ahs import AtomArrangement, DrivingField, AnalogHamiltonianSimulation
from braket.timings.time_series import TimeSeries
from braket.aws import AwsDevice  # imported only to show the submission API; never called unless RUN_ON_AWS

# Aquila's device ARN is just a string until you hand it to AwsDevice(...).
AQUILA_ARN = "arn:aws:braket:us-east-1::device/qpu/quera/Aquila"

# Cost discipline (project rule): keep the real submission OFF by default.
RUN_ON_AWS = False

print("braket.ahs imported; we will construct an AHS program locally (no AWS calls).")
print(f"Aquila device ARN: {AQUILA_ARN}")
print(f"RUN_ON_AWS = {RUN_ON_AWS}")

## 1. The atom geometry (register)

We lay out **five atoms on a regular pentagon ring**. Aquila works in physical units: coordinates are in **meters**, and realistic atom spacings are a few micrometers. With the right spacing, each atom is within the blockade radius of its two ring neighbors but *not* its two non-neighbors -- so the blockade graph is a 5-cycle ($C_5$), a small problem whose MIS we can verify by hand.

We place the ring with radius `6 um` and read off the nearest-neighbor and next-nearest-neighbor distances; the blockade radius will sit between them.

In [ ]:
N = 5
ring_radius = 6.0e-6  # meters (6 micrometers)

# Evenly spaced points on a circle; the +pi/2 just puts atom 0 at the top.
angles = np.array([2 * np.pi * k / N + np.pi / 2 for k in range(N)])
coords = np.stack([ring_radius * np.cos(angles), ring_radius * np.sin(angles)], axis=1)

print("Atom coordinates (micrometers):")
for k, (x, y) in enumerate(coords):
    print(f"  atom {k}: ({x * 1e6:6.3f}, {y * 1e6:6.3f})")

nn = np.linalg.norm(coords[0] - coords[1])   # adjacent on the ring
nnn = np.linalg.norm(coords[0] - coords[2])  # skip one
print(f"\nnearest-neighbor spacing:      {nn * 1e6:6.3f} um")
print(f"next-nearest-neighbor spacing: {nnn * 1e6:6.3f} um")

# Sanity: on a pentagon, adjacent atoms are strictly closer than skip-one atoms.
assert nn < nnn, "ring geometry should make neighbors closer than non-neighbors"

## 2. Blockade graph and the classical MIS

Choose the blockade radius $R_b$ to sit **between** the nearest- and next-nearest spacings. Then exactly the five ring edges are "blockaded": each pair of adjacent atoms cannot both be excited. That is the 5-cycle graph $C_5$.

We brute-force its Maximum Independent Set in numpy (only $2^5 = 32$ bitstrings) so we have a ground-truth answer to compare against the analog program. For a 5-cycle the MIS has size 2 -- you can pick at most every other atom around an odd ring.

In [ ]:
blockade_radius = 0.5 * (nn + nnn)  # between neighbor and non-neighbor spacing
print(f"blockade radius R_b = {blockade_radius * 1e6:.3f} um")

# Edge if the pair is within R_b: they cannot both be Rydberg-excited.
edges = []
for i, j in combinations(range(N), 2):
    if np.linalg.norm(coords[i] - coords[j]) <= blockade_radius:
        edges.append((i, j))
print("blockade edges (pairs that cannot both be |r>):", edges)
assert len(edges) == 5, "the pentagon should yield exactly the 5 ring edges (a 5-cycle)"

# Brute-force Maximum Independent Set over all 2^N selections.
best_set, best_size = [], -1
for bits in product([0, 1], repeat=N):
    independent = all(not (bits[i] and bits[j]) for (i, j) in edges)
    if independent:
        chosen = [k for k in range(N) if bits[k]]
        if len(chosen) > best_size:
            best_size, best_set = len(chosen), chosen

print(f"classical MIS size = {best_size}, e.g. atoms {best_set}")
# The maximum independent set of an odd cycle C_5 is floor(5/2) = 2.
assert best_size == 2, "MIS of a 5-cycle must be 2"

## 3. Visualize the register and its blockade graph

Atoms in the MIS are highlighted; grey lines are blockade edges. The analog machine's job is to find this set physically -- by evolving atoms toward the highest-occupancy independent configuration.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for (i, j) in edges:
    ax.plot(
        [coords[i, 0] * 1e6, coords[j, 0] * 1e6],
        [coords[i, 1] * 1e6, coords[j, 1] * 1e6],
        "-", color="0.75", zorder=1,
    )
in_mis = set(best_set)
for k, (x, y) in enumerate(coords):
    ax.scatter(
        x * 1e6, y * 1e6, s=420, zorder=3, edgecolors="k",
        c="#d1495b" if k in in_mis else "#3a6ea5",
    )
    ax.annotate(str(k), (x * 1e6, y * 1e6), ha="center", va="center", color="w", zorder=4)
ax.set_xlabel("x (um)")
ax.set_ylabel("y (um)")
ax.set_title("Pentagon register: red = a Maximum Independent Set")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 4. The time-dependent driving field

The global drive has three knobs, each a `TimeSeries` over the protocol duration `t_max`:

- **Amplitude** $\Omega(t)$ -- the Rabi frequency coupling $\ket{g}\leftrightarrow\ket{r}$ (rad/s). We ramp it up, hold, and ramp down so the atoms start and end idle in $\ket{g}$.
- **Detuning** $\delta(t)$ -- swept from **negative to positive**. This is the adiabatic trick: at large negative $\delta$ the ground state has *no* excitations; as $\delta$ crosses positive, exciting atoms becomes favorable, but the blockade caps how many neighbors can light up at once. Sweep slowly and the system tracks the instantaneous ground state straight into a Maximum Independent Set.
- **Phase** $\phi(t)$ -- held at 0 here (we do not need a phase profile for this protocol).

Values below are in Aquila-realistic ranges (a few Mrad/s, microsecond timescales). We print the full schedule so you can see real local output.

In [ ]:
t_max = 4.0e-6           # total evolution time: 4 microseconds
Omega_max = 1.5e7        # peak Rabi frequency ~15 Mrad/s
delta_start, delta_end = -2.0e7, 2.0e7  # detuning swept negative -> positive (rad/s)
ramp = 0.25e-6           # amplitude ramp up/down time

# Amplitude: 0 -> Omega_max (ramp up), hold, -> 0 (ramp down). Atoms idle in |g> at both ends.
amplitude = (
    TimeSeries()
    .put(0.0, 0.0)
    .put(ramp, Omega_max)
    .put(t_max - ramp, Omega_max)
    .put(t_max, 0.0)
)
# Detuning: linear sweep from negative to positive -- the adiabatic ramp that prepares the MIS.
detuning = TimeSeries().put(0.0, delta_start).put(t_max, delta_end)
# Phase: flat at 0.
phase = TimeSeries().put(0.0, 0.0).put(t_max, 0.0)

print("Amplitude (Rabi frequency) schedule  [time -> Omega]:")
for t, v in zip(amplitude.times(), amplitude.values()):
    print(f"  t = {t * 1e6:5.2f} us   Omega = {v / 1e6:6.2f} Mrad/s")
print("Detuning schedule  [time -> delta]:")
for t, v in zip(detuning.times(), detuning.values()):
    print(f"  t = {t * 1e6:5.2f} us   delta = {v / 1e6:6.2f} Mrad/s")

# The protocol must start and end with the drive off (atoms in |g>)...
assert amplitude.values()[0] == 0.0 and amplitude.values()[-1] == 0.0
# ...and the detuning must cross from negative to positive for the adiabatic sweep.
assert detuning.values()[0] < 0 < detuning.values()[-1]

## 5. Assemble the `AnalogHamiltonianSimulation` program

Now we build the real SDK objects: an `AtomArrangement` (via `.add(...)` for each atom), a `DrivingField` from the three time series, and the `AnalogHamiltonianSimulation(register=..., hamiltonian=...)`. This is exactly what you would submit to Aquila -- but constructing and inspecting it touches no network. We print the register coordinates read back **from the program object** and confirm it carries one Hamiltonian term (the single global drive).

In [ ]:
register = AtomArrangement()
for (x, y) in coords:
    register.add((float(x), float(y)))  # coordinates in meters

drive = DrivingField(amplitude=amplitude, phase=phase, detuning=detuning)
ahs_program = AnalogHamiltonianSimulation(register=register, hamiltonian=drive)

print(f"register atoms:      {len(ahs_program.register)}")
print(f"hamiltonian terms:   {len(ahs_program.hamiltonian.terms)}  (one global driving field)")
print("\nregister site coordinates read back from the program object:")
for k, site in enumerate(ahs_program.register):
    cx, cy = site.coordinate
    print(f"  site {k}: ({float(cx) * 1e6:6.3f}, {float(cy) * 1e6:6.3f}) um   [{site.site_type}]")

assert len(ahs_program.register) == N
assert len(ahs_program.hamiltonian.terms) == 1  # exactly the one DrivingField
print("\nProgram constructed locally and inspected -- no AWS contact.")

## 6. Submitting to Aquila (guarded)

Running `ahs_program` on Aquila is a **single line**: `AwsDevice(AQUILA_ARN).run(ahs_program, shots=...)`. We keep it behind `RUN_ON_AWS`, which is `False`, so this notebook never instantiates `AwsDevice`, never queries device properties, and never submits a task.

**Cost note (per the project's cost rule).** QuEra Aquila is metered like other QPUs: roughly **$0.30 per task** plus **$0.01 per shot**. A typical 100-shot run is about `$0.30 + 100 x $0.01 = $1.30`, before any queue wait. Aquila is also region- and window-limited (availability windows, max ~256 atoms, geometry constraints checked against `device.properties`). Only flip `RUN_ON_AWS = True` with valid AWS credentials, after validating the geometry and schedule -- which is exactly what we did above for free.

In [ ]:
if RUN_ON_AWS:
    # COST: ~$0.30 per task + $0.01 per shot. Requires AWS credentials and Aquila availability.
    device = AwsDevice(AQUILA_ARN)
    n_shots = 100
    # (Optionally validate against device.properties / paradigm constraints before submitting.)
    task = device.run(ahs_program, shots=n_shots)
    result = task.result()
    # Each shot reports per-site status (empty / |g> ground / |r> Rydberg);
    # Rydberg sites approximate an independent set, and the most common one ~ the MIS.
    print(result.measurements[:3])
else:
    print("RUN_ON_AWS is False -- no AwsDevice created, no task submitted, no charge incurred.")
    print("The program above is fully built and validated locally; the classical MIS is the")
    print(f"expected answer: size {best_size} (e.g. atoms {best_set}).")

## Exercises

```python
# TODO 1: Change the geometry. Replace the 5-atom ring with a 6-atom ring (hexagon, C_6).
#         Re-run the blockade-graph + MIS cells. An even cycle's MIS is N/2 -- confirm you get 3.

# TODO 2: Tune the blockade. Shrink ring_radius (e.g. 4.5 um) so that next-nearest atoms ALSO
#         fall inside R_b. How does the edge set change, and what does that do to the MIS size?

# TODO 3: Add a 6th atom in the center of the pentagon. Recompute edges and MIS. A hub
#         connected to everything cannot be in any 2-atom independent set -- verify numerically.

# TODO 4: Slow the sweep. Double t_max (keeping the same start/end detuning). Adiabatic theory
#         says a slower sweep tracks the ground state better. Print the new schedule and reason
#         about why a slower delta-ramp should improve the MIS success probability on hardware.

# TODO 5 (AWS, optional): With credentials and RUN_ON_AWS=True, before submitting, instantiate
#         AwsDevice(AQUILA_ARN) and inspect device.properties.paradigm -- the max atom count,
#         lattice area, and min atom spacing. Assert your register satisfies them. (Reading
#         properties is free, but the .run() call is metered -- estimate first.)
```

## Summary

- **Aquila is analog, not gate-model.** You specify a *geometry* of neutral atoms and a *time-dependent drive* (Rabi frequency $\Omega(t)$, detuning $\delta(t)$, phase $\phi(t)$), then evolve under the Rydberg Hamiltonian -- no circuit.
- **The Rydberg blockade encodes a graph.** Atoms within the blockade radius $R_b$ cannot both reach $\ket{r}$, so excited atoms form an *independent set*; an adiabatic detuning sweep (negative to positive) drives the array toward a **Maximum Independent Set**.
- **We built the real program with `braket.ahs`** -- `AtomArrangement.add(...)`, `DrivingField`, `AnalogHamiltonianSimulation(register=..., hamiltonian=...)` -- and inspected its register coordinates and one Hamiltonian term locally, with **no AWS calls**.
- **A 5-atom pentagon is the 5-cycle $C_5$,** whose MIS (size 2) we verified classically in numpy as ground truth for the analog answer.
- **Cost discipline held:** the only line that would spend money, `AwsDevice(AQUILA_ARN).run(ahs_program, shots=...)`, stayed behind `RUN_ON_AWS = False` with a `$0.30/task + $0.01/shot` estimate -- local-first, exactly the workflow the module preaches.

**Next:** [`05-simulator-comparison.ipynb`](05-simulator-comparison.ipynb) -- run the same gate circuit across the simulator ladder (Local, SV1, DM1, TN1) and weigh accuracy, runtime, and cost.